# Estratégia de Retenção de Clientes: Análise de Rotatividade Model Fitness
## 1 Introdução
A rede de academias Model Fitness está focada em digitalizar os perfis de seus clientes para combater a rotatividade. O desafio principal é que muitos clientes "saem de fininho", parando de frequentar sem cancelar formalmente o contrato. Para esta análise, consideramos que um cliente deu churn se não apareceu na unidade por um mês inteiro. 
### 1.1 Objetivos do Projeto
O objetivo deste estudo é analisar os dados dos clientes e desenvolver uma estratégia de retenção baseada em dados.  As etapas principais incluem:
- **Análise Exploratória de Dados (AED)**: Estudar o perfil dos clientes que ficam versus os que saem.
- **Modelagem Preditiva**: Treinar modelos de classificação binária para prever a probabilidade de rotatividade para o mês seguinte.
- **Agrupamento (Clustering)**: Identificar grupos (clusters) de usuários típicos para entender seus comportamentos e lealdade.
- **Recomendações**: Propor ações de marketing e mudanças no serviço para diminuir a rotatividade.
### 1.2 Descrição dos Dados
O conjunto de dados contém informações sobre o mês atual (churn) e o histórico do mês anterior, incluindo:
- **Perfil**: Gênero, proximidade da unidade, parceria com empresas, indicação de amigos e telefone.
- **Uso**: Idade, tempo de vida do cliente (lifetime), período do contrato, frequência de visitas e gastos adicionais.
## 2 Importação de Bibliotecas e Carregando Dados

In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans

# Carregando o conjunto de dados
# O caminho do arquivo foi especificado nas instruções do projeto [cite: 51]
try:
    df_gym_churn = pd.read_csv('datasets/gym_churn_us.csv')
    print("Dados carregados com sucesso!")
except FileNotFoundError:
    print("Erro: Arquivo não encontrado. Verifique o caminho do dataset.")

# Visualização inicial para confirmar se os dados estão corretos
display(df_gym_churn.info())
display(df_gym_churn.head())



Dados carregados com sucesso!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   gender                             4000 non-null   int64  
 1   Near_Location                      4000 non-null   int64  
 2   Partner                            4000 non-null   int64  
 3   Promo_friends                      4000 non-null   int64  
 4   Phone                              4000 non-null   int64  
 5   Contract_period                    4000 non-null   int64  
 6   Group_visits                       4000 non-null   int64  
 7   Age                                4000 non-null   int64  
 8   Avg_additional_charges_total       4000 non-null   float64
 9   Month_to_end_contract              4000 non-null   float64
 10  Lifetime                           4000 non-null   int64  
 11  Avg_class_frequency_total 

None

,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
0,1,1,1,1,0,6,1,29,14.227470,5.0,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12.0,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1.0,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12.0,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1.0,3,1.113884,1.120078,0


### 2.1 Padronizando Informações
Aqui modificaremos os nomes das colunas para padrão snake_case.

In [8]:
# Convertendo os nomes das colunas para snake_case (letras minúsculas)
df_gym_churn.columns = [i.lower() for i in df_gym_churn.columns]

# Visualizando as primeiras linhas com o novo padrão
df_gym_churn.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   gender                             4000 non-null   int64  
 1   near_location                      4000 non-null   int64  
 2   partner                            4000 non-null   int64  
 3   promo_friends                      4000 non-null   int64  
 4   phone                              4000 non-null   int64  
 5   contract_period                    4000 non-null   int64  
 6   group_visits                       4000 non-null   int64  
 7   age                                4000 non-null   int64  
 8   avg_additional_charges_total       4000 non-null   float64
 9   month_to_end_contract              4000 non-null   float64
 10  lifetime                           4000 non-null   int64  
 11  avg_class_frequency_total          4000 non-null   float